# Vmin Starting-Point Analysis (All UPSVF Domains)
Generated: 2026-06-16 19:46:49

This notebook is a starting point with your die mapping and target rules:
- Cdie = U5 (AT/CR/CCF/CLR domains), T0 target S2T=0.87, filter DevRevStep_SORT_U1.U5=8PY6CVT
- GT die = U4 (GT/GTVPG), A-step target S2T=0.85, filter DevRevStep_SORT_U1.U4=8PQ8CVAA
- Hub die = U2 (all other domains), A-step target S2T=0.85, filter DevRevStep_SORT_U1.U2=8PF9CVA

Input file:
- R:\\Products\\NVL\\NVL-H\\Weekly Runs\\Vmin_NVLHM66A0H30N00S623_WW25_2026_merged.csv

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
csv_path = r'R:\\Products\\NVL\\NVL-H\\Weekly Runs\\Vmin_NVLHM66A0H30N00S623_WW25_2026_merged.csv'
df = pd.read_csv(csv_path, low_memory=False)
u5_col = 'UPM_0107_DPMH156P48ULVTINVD4_FULLDIE_0950_MED_SORT_U1.U5'
u2_col = 'TPI_UPM::UPM_X_SCREEN_K_START_X_X_X_X_FEM_CALC_UPM_FEM_ULVT_FULLDIE_PDK0P9_0950MV_SORT_U1.U2'
u4_col = 'TPI_UPM::UPM_X_SCREEN_K_START_X_X_X_X_FEM_CALC_UPM_FEM_ULVT_FULLDIE_PDK0P9_0950MV_SORT_U1.U4'
df['UPM_0107_DPMH156P48ULVTINVD4_FULLDIE_0950_MED_SORT_U1.U5 S2T'] = pd.to_numeric(df[u5_col], errors='coerce') / 9154.0
vmin_cols = [c for c in df.columns if ('UPSVFPASSFLOW' in c and '_RCS_' in c)]
len(vmin_cols)

In [ ]:
def die_cfg(c):
    if '_U1PU5_' in c:
        return {'die':'CDIE','x':'UPM_0107_DPMH156P48ULVTINVD4_FULLDIE_0950_MED_SORT_U1.U5 S2T','dev':'DevRevStep_SORT_U1.U5','step':'8PY6CVT','target':0.87}
    if '_U1PU4_' in c:
        return {'die':'GTDIE','x':'TPI_UPM::UPM_X_SCREEN_K_START_X_X_X_X_FEM_CALC_UPM_FEM_ULVT_FULLDIE_PDK0P9_0950MV_SORT_U1.U4','dev':'DevRevStep_SORT_U1.U4','step':'8PQ8CVAA','target':0.85}
    if '_U1PU2_' in c:
        return {'die':'HUBDIE','x':'TPI_UPM::UPM_X_SCREEN_K_START_X_X_X_X_FEM_CALC_UPM_FEM_ULVT_FULLDIE_PDK0P9_0950MV_SORT_U1.U2','dev':'DevRevStep_SORT_U1.U2','step':'8PF9CVA','target':0.85}
    return None

rows = []
for c in vmin_cols:
    m = re.search(r"_RCS_([A-Z0-9]+)_([0-9]+\\.[0-9]+)_([0-9]+)$", c)
    if not m: continue
    dom, freq, core = m.group(1), float(m.group(2)), int(m.group(3))
    cfg = die_cfg(c)
    if not cfg: continue
    x = pd.to_numeric(df[cfg["x"]], errors="coerce")
    y = pd.to_numeric(df[c], errors="coerce")
    d = (df[cfg["dev"]] == cfg["step"])
    msk = d & x.notna() & y.notna()
    if msk.sum() < 10: continue
    x2 = x[msk].values
    y2 = y[msk].values
    slope, intercept = np.polyfit(x2, y2, 1)
    yhat = slope*x2 + intercept
    ss_res = np.sum((y2-yhat)**2)
    ss_tot = np.sum((y2-np.mean(y2))**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    rows.append({"Die":cfg["die"],"Domain":dom,"Freq":freq,"Core":core,"Column":c,"N":int(msk.sum()),"Slope":slope,"Intercept":intercept,"R2":r2,"Target":cfg["target"],"Vmin_at_target":slope*cfg["target"]+intercept})
fit_df = pd.DataFrame(rows).sort_values(["Die","Domain","Freq","Core"])
fit_df.head()

In [ ]:
vf_df = (fit_df.dropna(subset=["Vmin_at_target"])
         .groupby(["Die","Domain","Freq"], as_index=False)
         .agg(Vmin=("Vmin_at_target","mean"), Contributors=("Vmin_at_target","size"))
         .sort_values(["Die","Domain","Freq"]))
vf_df.head(30)

In [ ]:
for d, g in vf_df.groupby("Domain"):
    if len(g) < 2: continue
    plt.figure(figsize=(6.5,4.5))
    plt.plot(g["Freq"], g["Vmin"], marker="o", linewidth=2)
    plt.title(f"{d} normalized VF curve")
    plt.xlabel("Frequency (GHz)")
    plt.ylabel("Vmin @ target")
    plt.grid(True, alpha=0.3)
    plt.show()

## Generated Summary Preview

```csv
"Die","Domain","Freq","Vmin","Contributors"
"CDIE","AT","1.2","0.374","2"
"CDIE","AT","2.8","0.6002","2"
"CDIE","AT","3.2","0.6718","2"
"CDIE","AT","3.6","0.7561","2"
"CDIE","AT","3.7","-4.5162","2"
"CDIE","AT","3.9","0.979","2"
"CDIE","AT","4.2","1.0664","2"
"CDIE","CCF","1.2","0.4675","1"
"CDIE","CCF","3","0.6669","1"
"CDIE","CCF","3.4","0.7403","1"
"CDIE","CCF","4","0.8611","1"
"CDIE","CCF","4.3","0.9696","1"
"CDIE","CCF","4.8","1.119","1"
"CDIE","CCF","5","1.1706","1"
"CDIE","CR","1.2","0.4804","2"
"CDIE","CR","3","0.6619","2"
"CDIE","CR","4.1","0.8521","2"
"CDIE","CR","4.5","0.9614","2"
"CDIE","CR","4.8","1.0659","2"
"CDIE","CR","5","1.1482","2"
"GTDIE","GT","1.2","0.5958","1"
"GTDIE","GT","1.5","0.6265","1"
"GTDIE","GT","2","0.6991","1"
"GTDIE","GT","2.2","0.7301","1"
"GTDIE","GT","2.9","0.8757","1"
"GTDIE","GTVPG","1.2","0.5958","1"
"GTDIE","GTVPG","1.5","0.6265","1"
"GTDIE","GTVPG","2","0.6991","1"
"GTDIE","GTVPG","2.2","0.7301","1"
"GTDIE","GTVPG","2.9","0.8757","1"
"HUBDIE","SAAT","1.2","0.5065","1"
"HUBDIE","SAAT","2.4","0.6956","1"
"HUBDIE","SAAT","3","0.8315","1"
"HUBDIE","SAAT","3.2","0.9746","1"
"HUBDIE","SAAT","3.6","1.0145","1"
"HUBDIE","SAC","1.4","0.5852","1"
"HUBDIE","SAC","2.45","0.8052","1"
"HUBDIE","SAC","2.7","0.8966","1"
"HUBDIE","SAC","2.8","0.9572","1"
"HUBDIE","SAC","3","1.0677","1"
```